In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# BlindDetection-V1 N_dev=256 calibration handoff

Prepared only. This notebook uses a detached reviewed producer exact and one formal calibration-runner attempt. It preserves a fixed 256-row engineering denominator and a science denominator of zero. Run only under a separate execution authorization after the reviewed exact has been pushed.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import torch

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
PRODUCER_EXACT = '0ce90f2f11669c4ab6e492cb196404bf2ff0401b'
CHECKOUT = Path('/content/cegwm-blind-detection-v1-calibration')
INPUT_ROOT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/calibration-input')
ROSTER = INPUT_ROOT / 'development-roster-256.json'
KEY_FILE = INPUT_ROOT / 'detection-key.bin'
CONFIG_FILE = INPUT_ROOT / 'calibration-config.json'
RUNS_ROOT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/calibration-runs')
RUN_ID = f"{PRODUCER_EXACT}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
RUN_ROOT = RUNS_ROOT / RUN_ID
if not torch.cuda.is_available():
    raise RuntimeError('GPU required; N_dev=256 was not executed')
if CHECKOUT.exists() or RUN_ROOT.exists():
    raise FileExistsError('fresh checkout and create-only run root required')
RUN_ROOT.mkdir(parents=True, exist_ok=False)
subprocess.run(['git', 'clone', REPO_URL, str(CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', PRODUCER_EXACT], check=True)
def git(*args):
    return subprocess.run(
        ['git', '-C', str(CHECKOUT), *args], check=True, capture_output=True, text=True,
    ).stdout.strip()
if git('rev-parse', 'HEAD') != PRODUCER_EXACT or git('branch', '--show-current') != '':
    raise RuntimeError('detached producer exact differs')
if git('status', '--porcelain=v1') != '':
    raise RuntimeError('producer checkout must be clean before installation')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(CHECKOUT)], check=True)
if git('rev-parse', 'HEAD') != PRODUCER_EXACT or git('status', '--porcelain=v1') != '':
    raise RuntimeError('producer exact or clean state changed during installation')


In [ ]:
if not all(path.is_file() for path in (ROSTER, KEY_FILE, CONFIG_FILE)):
    raise FileNotFoundError('fixed roster, key file, or calibration config is absent')
sys.path.insert(0, str(CHECKOUT))
from experiments import run_blind_detection_v1 as calibration_runner
from cegwm.shared.keys import normalize_detection_key, public_key_digest

roster, disjoint_evidence, roster_file_sha256 = calibration_runner.load_roster_inputs(ROSTER)
config = json.loads(CONFIG_FILE.read_text(encoding='utf-8'))
if not isinstance(config, dict) or set(config) != {'runtime_factory'}:
    raise ValueError('calibration config fields differ')
runtime_factory = config['runtime_factory']
if not isinstance(runtime_factory, str) or runtime_factory.count(':') != 1:
    raise ValueError('runtime factory specification differs')
key_digest = public_key_digest(normalize_detection_key(KEY_FILE.read_bytes()))
INPUT_SUMMARY = {
    'config_file_sha256': hashlib.sha256(CONFIG_FILE.read_bytes()).hexdigest(),
    'denominator': len(roster.units),
    'disjoint_evidence_digest': hashlib.sha256(
        calibration_runner.stable_json_bytes(dict(sorted(disjoint_evidence.items())))
    ).hexdigest(),
    'key_public_digest': key_digest,
    'producer_exact': PRODUCER_EXACT,
    'roster_digest': roster.digest,
    'roster_file_sha256': roster_file_sha256,
}
print('CEGWM_BLIND_CALIBRATION_INPUTS ' + json.dumps(INPUT_SUMMARY, sort_keys=True))
if 'CALIBRATION_RUNNER_CALLS' not in globals():
    CALIBRATION_RUNNER_CALLS = 0


In [ ]:
assert CALIBRATION_RUNNER_CALLS == 0
RESULT = RUN_ROOT / 'calibration_result.json'
THRESHOLD = RUN_ROOT / 'blind_detection_v1_thresholds.json'
STDOUT = RUN_ROOT / 'runner.stdout.txt'
STDERR = RUN_ROOT / 'runner.stderr.txt'
STATUS = RUN_ROOT / 'notebook_status.json'
MANIFEST = RUN_ROOT / 'artifact_manifest.json'
command = [
    sys.executable, str(CHECKOUT / 'experiments/run_blind_detection_v1.py'),
    'calibrate-and-freeze', '--roster', str(ROSTER), '--key-file', str(KEY_FILE),
    '--runtime-factory', runtime_factory, '--producer-exact', PRODUCER_EXACT,
    '--output', str(THRESHOLD), '--result-output', str(RESULT),
]
CALIBRATION_RUNNER_CALLS += 1
with STDOUT.open('xb') as stdout, STDERR.open('xb') as stderr:
    completed = subprocess.run(command, cwd=CHECKOUT, stdout=stdout, stderr=stderr, check=False)
if CALIBRATION_RUNNER_CALLS != 1 or not RESULT.is_file():
    with STATUS.open('xb') as sink:
        sink.write(calibration_runner.stable_json_bytes({
            'producer_exact': PRODUCER_EXACT, 'runner_rc': completed.returncode,
            'status': 'OPERATIONAL_BLOCKED_RESULT_ABSENT',
        }))
    raise RuntimeError('OPERATIONAL_BLOCKED: formal runner result is absent')
result = json.loads(RESULT.read_text(encoding='ascii'))
success = (
    completed.returncode == 0
    and result.get('status') == 'THRESHOLD_FROZEN_AFTER_0_OF_256_FRESH_REPLAY'
    and result.get('fresh_replay_zero_of_256') is True
    and THRESHOLD.is_file()
)
if not success and THRESHOLD.exists():
    raise RuntimeError('threshold exists without a complete successful fresh replay')
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
artifacts = {
    path.name: {'sha256': sha256(path), 'size': path.stat().st_size}
    for path in (RESULT, STDOUT, STDERR)
}
if THRESHOLD.is_file():
    artifacts[THRESHOLD.name] = {'sha256': sha256(THRESHOLD), 'size': THRESHOLD.stat().st_size}
manifest = {
    'artifacts': artifacts,
    'claim_ceiling': 'engineering_N_dev_256_threshold_calibration_only; science_denominator=0',
    'input_summary': INPUT_SUMMARY,
    'producer_exact': PRODUCER_EXACT,
    'runner_rc': completed.returncode,
    'runner_status': result.get('status'),
    'schema_version': 'cegwm_blind_detection_v1_calibration_manifest_v1',
}
with MANIFEST.open('xb') as sink:
    sink.write(calibration_runner.stable_json_bytes(manifest))
with STATUS.open('xb') as sink:
    sink.write(calibration_runner.stable_json_bytes({
        'manifest_sha256': sha256(MANIFEST), 'producer_exact': PRODUCER_EXACT,
        'runner_rc': completed.returncode, 'status': result.get('status'),
    }))
if not success:
    raise RuntimeError('calibration did not freeze a threshold; inspect retained records')


In [ ]:
stored_manifest = json.loads(MANIFEST.read_text(encoding='ascii'))
if stored_manifest.get('producer_exact') != PRODUCER_EXACT:
    raise RuntimeError('stored producer exact differs')
for name, record in stored_manifest['artifacts'].items():
    artifact = RUN_ROOT / name
    if not artifact.is_file() or sha256(artifact) != record['sha256']:
        raise RuntimeError('stored artifact readback differs')
print('CEGWM_BLIND_CALIBRATION_READBACK ' + json.dumps({
    'artifact_manifest_sha256': sha256(MANIFEST),
    'producer_exact': PRODUCER_EXACT,
    'runner_status': stored_manifest['runner_status'],
}, sort_keys=True))
